# Orquestador con modelo Gemini

### Cargar las librerías necesarias

In [5]:
!pip install langgraph langchain langchain-core langchain-community
!pip install google-generativeai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 21.2 MB/s eta 0:00:00


### Cargar la API Key y el modelo

In [6]:
import os
import google.generativeai as genai

# Configura tu clave de la API
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# Cliente Gemini 1.5 Flash
model_gemini = genai.GenerativeModel("gemini-1.5-flash")


### Definición de las simulaciones de las tools

In [7]:
def tool_ocr(input_data):
    print("[TOOL] OCR ejecutado")
    return {"ocr_result": "Texto leído en la imagen"}

def tool_object_detection(input_data):
    print("[TOOL] Object Detection ejecutado")
    return {"objects": ["perro", "bicicleta"]}

def tool_describe_scene(input_data):
    print("[TOOL] Describe Scene ejecutado")
    return {"description": "Una calle concurrida con gente caminando"}

TOOLS = {
    "ocr": tool_ocr,
    "object_detection": tool_object_detection,
    "describe_scene": tool_describe_scene,
}

### Prompt del orquestador

In [29]:
def orchestrator_prompt(user_input, tool_list):
    return f"""
Eres un orquestador de una aplicación que ayuda a personas con discapacidad visual que decide qué herramientas usar para responder al usuario.

Herramientas disponibles: {tool_list}

'ocr': permite leer texto en una imagen.
'object_detection': permite detectar objetos en una imagen.
'describe_scene': permite describir la escena en una imagen.
Devuelve EXCLUSIVAMENTE un JSON válido con esta estructura:
{{
  "tools": ["..."],
  "justification": "..."
}}
Donde "tools" debe ser una lista con las herramientas a ejecutar en el orden que se deben ejecutar
y "justification" es un string que justifica brevemente en una frase de menos de 15 palabras esta elección.

Petición:
"{user_input}"
"""


### Nodos principales

In [26]:
import re
import json

def parse_gemini_response(response):
    """
    Extrae y parsea JSON desde una respuesta de Gemini que puede incluir ```json ... ```
    """
    text = response.text.strip()

    # Si viene dentro de bloque markdown ```json ... ```
    if "```" in text:
        match = re.search(r"```json\s*(.*?)\s*```", text, re.DOTALL)
        if match:
            text = match.group(1)

    # Parseamos el JSON
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print("[ERROR] No se pudo parsear JSON. Texto devuelto:")
        print(text)
        return {"tools": [], "order": [], "justification": "Error parsing JSON"}


In [27]:
import json
import time
import re

def orchestrator(user_input):
    prompt = orchestrator_prompt(user_input, list(TOOLS.keys()))
    response = model_gemini.generate_content(prompt)
    return parse_gemini_response(response)

def execute_tools(user_input, tool_order):
    outputs = {}
    for tool_name in tool_order:
        if tool_name in TOOLS:
            outputs[tool_name] = TOOLS[tool_name](user_input)
        else:
            outputs[tool_name] = {"error": f"Tool {tool_name} no disponible"}
    return outputs

def response_generator(user_input, tools_outputs):
    prompt = f"""
Usuario: "{user_input}"
Resultados de las herramientas: {tools_outputs}

Genera una respuesta final clara para el usuario.
"""
    response = model_gemini.generate_content(prompt)
    return response.text

def validator(user_input, final_response):
    # Simulación de validación (ej: si contiene palabra clave del input)
    return any(word.lower() in final_response.lower() for word in user_input.split())

def logger(user_input, orchestrator_json, tools_outputs, final_response, file="logs.json"):
    log_entry = {
        "timestamp": time.time(),
        "user_input": user_input,
        "orchestrator_json": orchestrator_json,
        "tools_outputs": tools_outputs,
        "final_response": final_response,
    }
    try:
        with open(file, "r") as f:
            logs = json.load(f)
    except:
        logs = []
    logs.append(log_entry)
    with open(file, "w") as f:
        json.dump(logs, f, indent=2, ensure_ascii=False)


### Pipeline completo de ejecutción

In [18]:
def run_pipeline(user_input, max_loops=2):
    loop = 0
    while loop < max_loops:
        orchestrator_json = orchestrator(user_input)
        print(orchestrator_json)
        tools_outputs = execute_tools(user_input, orchestrator_json.get("order", []))
        final_response = response_generator(user_input, tools_outputs)

        if validator(user_input, final_response):
            print("[VALIDADO] Respuesta aceptada")
            logger(user_input, orchestrator_json, tools_outputs, final_response)
            return final_response
        else:
            print("[VALIDACIÓN FALLIDA] Reintentando...")
            loop += 1

    print("[STOP] Se alcanzó el máximo de iteraciones")
    logger(user_input, orchestrator_json, tools_outputs, final_response)
    return final_response


In [30]:
user_input = "¿Qué pone en el cartel?"
respuesta = run_pipeline(user_input)
print("RESPUESTA FINAL:", respuesta)

{'tools': ['ocr'], 'justification': 'La petición requiere leer texto de una imagen.'}
[VALIDADO] Respuesta aceptada
RESPUESTA FINAL: No se puede ver el cartel.  Necesitaría una imagen o una descripción del cartel para poder decirte qué pone en él.

